# Exercise 4

Write a discrete event simulation program for a blocking system, i.e. a system with $m$ service units and no waiting room. The offered traffic $A$ is the product of the mean arrival rate and the mean service time.

## Part 1

The arrival process is modeled as a Poisson process. Report the fraction of blocked customers, and a confidence interval for this fraction. Choose the service time distribution as exponential.

Parameters: $m = 10$, mean service time is $8$ time units, mean time between customers is $1$ time unit corresponding to an offered traffic of $8$ Erlang, $10 \times 10.000$ customers.

This system is sufficiently simple such that the analytical solution is known. See the last slide for the solution. Verify your simulation program using this knowledge.

In [2]:
import numpy as np
import math
from scipy.stats import t

# Parameters
m = 10
mean_interarrival = 1
mean_service = 8
n_customers = 10000
n_replications = 10

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system(n_customers, m, mean_interarrival, mean_service, rng):
    """
    Simulates a blocking system with m service units and no waiting room.
    Arrivals are generated from exponential inter-arrival times.
    Service times are exponentially distributed.
    """

    # Generate arrival times
    interarrival_times = rng.exponential(mean_interarrival, size=n_customers)
    arrival_times = np.cumsum(interarrival_times)

    # Generate service times
    service_times = rng.exponential(mean_service, size=n_customers)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If this server is free when the customer arrives, serve the customer
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    blocking_fraction = blocked / n_customers

    return blocking_fraction


# Run 10 independent replications
blocking_fractions = []

for i in range(n_replications):
    fraction = simulate_blocking_system(
        n_customers,
        m,
        mean_interarrival,
        mean_service,
        rng
    )
    blocking_fractions.append(fraction)

blocking_fractions = np.array(blocking_fractions)

# Estimate mean blocking fraction
mean_blocking = np.mean(blocking_fractions)

# Confidence interval based on the 10 replications
sample_sd = np.std(blocking_fractions, ddof=1)
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

# Erlang B formula for comparison
A = mean_service / mean_interarrival

def erlang_b(A, m):
    numerator = A**m / math.factorial(m)
    denominator = sum(A**i / math.factorial(i) for i in range(m + 1))
    return numerator / denominator

exact_blocking = erlang_b(A, m)

print("Blocking fractions from the 10 replications:")
print(blocking_fractions)

print("\nEstimated blocking fraction:")
print(mean_blocking)

print("\n95% confidence interval:")
print((lower, upper))

print("\nExact Erlang B blocking probability:")
print(exact_blocking)

Blocking fractions from the 10 replications:
[0.119  0.1222 0.1284 0.1294 0.1247 0.123  0.1156 0.1303 0.1228 0.1201]

Estimated blocking fraction:
0.12355000000000001

95% confidence interval:
(np.float64(0.1201515318151554), np.float64(0.12694846818484462))

Exact Erlang B blocking probability:
0.1216610642529515


## Part 2

The arrival process is modeled as a renewal process using the same parameters as in Part 1 when possible. Report the fraction of blocked customers, and a confidence interval for this fraction for at least the following two cases:

### a) Experiment with Erlang distributed inter-arrival times. The Erlang distribution should have a mean of $1$ time unit.

In [3]:
import numpy as np
from scipy.stats import t

# Parameters
m = 10
mean_interarrival = 1
mean_service = 8
n_customers = 10000
n_replications = 10

# Erlang parameter
erlang_shape = 2
erlang_scale = mean_interarrival / erlang_shape

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system_erlang_arrivals(
    n_customers, m, erlang_shape, erlang_scale, mean_service, rng
):
    """
    Simulates a blocking system with Erlang distributed inter-arrival times.
    The system has m service units and no waiting room.
    """

    # Generate Erlang inter-arrival times
    interarrival_times = rng.gamma(
        shape=erlang_shape,
        scale=erlang_scale,
        size=n_customers
    )

    arrival_times = np.cumsum(interarrival_times)

    # Generate exponential service times
    service_times = rng.exponential(mean_service, size=n_customers)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If the server is free, the customer is served
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    return blocked / n_customers


# Run 10 independent replications
blocking_fractions = []

for i in range(n_replications):
    fraction = simulate_blocking_system_erlang_arrivals(
        n_customers,
        m,
        erlang_shape,
        erlang_scale,
        mean_service,
        rng
    )
    blocking_fractions.append(fraction)

blocking_fractions = np.array(blocking_fractions)

# Estimate mean blocking fraction
mean_blocking = np.mean(blocking_fractions)

# Confidence interval
sample_sd = np.std(blocking_fractions, ddof=1)
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

print("Blocking fractions from the 10 replications:")
print(blocking_fractions)

print("\nEstimated blocking fraction:")
print(mean_blocking)

print("\n95% confidence interval:")
print((lower, upper))

Blocking fractions from the 10 replications:
[0.0929 0.0945 0.0856 0.0889 0.0864 0.0973 0.0906 0.0904 0.0928 0.0913]

Estimated blocking fraction:
0.09107000000000001

95% confidence interval:
(np.float64(0.08852312178215921), np.float64(0.09361687821784082))


### b) Hyperexponential inter-arrival times. The parameters for the hyperexponential distribution should be $p_1 = 0.8$, $\lambda_1 = 0.8333$, $p_2 = 0.2$, $\lambda_2 = 5.0$.

In [4]:
import numpy as np
from scipy.stats import t

# Parameters
m = 10
mean_service = 8
n_customers = 10000
n_replications = 10

# Hyperexponential parameters
p1 = 0.8
lambda1 = 0.8333
p2 = 0.2
lambda2 = 5.0

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system_hyperexp_arrivals(
    n_customers, m, p1, lambda1, p2, lambda2, mean_service, rng
):
    """
    Simulates a blocking system with hyperexponential inter-arrival times.
    The system has m service units and no waiting room.
    """

    # Choose which exponential distribution each inter-arrival time comes from
    choices = rng.random(n_customers)

    interarrival_times = np.zeros(n_customers)

    # With probability p1, use exponential distribution with rate lambda1
    mask1 = choices < p1
    interarrival_times[mask1] = rng.exponential(
        scale=1 / lambda1,
        size=np.sum(mask1)
    )

    # With probability p2, use exponential distribution with rate lambda2
    mask2 = ~mask1
    interarrival_times[mask2] = rng.exponential(
        scale=1 / lambda2,
        size=np.sum(mask2)
    )

    # Arrival times
    arrival_times = np.cumsum(interarrival_times)

    # Exponential service times
    service_times = rng.exponential(mean_service, size=n_customers)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If the server is free, the customer is served
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    return blocked / n_customers


# Run 10 independent replications
blocking_fractions = []

for i in range(n_replications):
    fraction = simulate_blocking_system_hyperexp_arrivals(
        n_customers,
        m,
        p1,
        lambda1,
        p2,
        lambda2,
        mean_service,
        rng
    )
    blocking_fractions.append(fraction)

blocking_fractions = np.array(blocking_fractions)

# Estimate mean blocking fraction
mean_blocking = np.mean(blocking_fractions)

# Confidence interval
sample_sd = np.std(blocking_fractions, ddof=1)
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

print("Blocking fractions from the 10 replications:")
print(blocking_fractions)

print("\nEstimated blocking fraction:")
print(mean_blocking)

print("\n95% confidence interval:")
print((lower, upper))

Blocking fractions from the 10 replications:
[0.1328 0.143  0.1276 0.1426 0.1363 0.144  0.1332 0.1313 0.1333 0.1409]

Estimated blocking fraction:
0.1365

95% confidence interval:
(np.float64(0.13239626428652), np.float64(0.14060373571348003))


## Part 3

The arrival process is again a Poisson process like in Part 1. Experiment with different service time distributions with the same mean service time and $m$ as in Part 1 and Part 2.

### a) Constant service time

In [5]:
import numpy as np
from scipy.stats import t
import math

# Parameters
m = 10
mean_interarrival = 1
constant_service_time = 8
n_customers = 10000
n_replications = 10

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system_constant_service(
    n_customers, m, mean_interarrival, constant_service_time, rng
):
    """
    Simulates a blocking system with Poisson arrivals and constant service times.
    The system has m service units and no waiting room.
    """

    # Poisson arrivals: exponential inter-arrival times
    interarrival_times = rng.exponential(mean_interarrival, size=n_customers)
    arrival_times = np.cumsum(interarrival_times)

    # Constant service times
    service_times = np.full(n_customers, constant_service_time)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If the server is free, the customer is served
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    return blocked / n_customers


# Run 10 independent replications
blocking_fractions = []

for i in range(n_replications):
    fraction = simulate_blocking_system_constant_service(
        n_customers,
        m,
        mean_interarrival,
        constant_service_time,
        rng
    )
    blocking_fractions.append(fraction)

blocking_fractions = np.array(blocking_fractions)

# Estimate mean blocking fraction
mean_blocking = np.mean(blocking_fractions)

# Confidence interval
sample_sd = np.std(blocking_fractions, ddof=1)
alpha = 0.05
t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

# Erlang B formula
A = constant_service_time / mean_interarrival

def erlang_b(A, m):
    numerator = A**m / math.factorial(m)
    denominator = sum(A**i / math.factorial(i) for i in range(m + 1))
    return numerator / denominator

exact_blocking = erlang_b(A, m)

print("Blocking fractions from the 10 replications:")
print(blocking_fractions)

print("\nEstimated blocking fraction:")
print(mean_blocking)

print("\n95% confidence interval:")
print((lower, upper))

print("\nExact Erlang B blocking probability:")
print(exact_blocking)

Blocking fractions from the 10 replications:
[0.1218 0.1253 0.1204 0.1215 0.1191 0.1176 0.1285 0.1178 0.1251 0.1195]

Estimated blocking fraction:
0.12165999999999999

95% confidence interval:
(np.float64(0.11909063910170868), np.float64(0.1242293608982913))

Exact Erlang B blocking probability:
0.1216610642529515


### b) Pareto distributed service times with at least $k = 1.05$ and $k = 2.05$.


In [6]:
import numpy as np
from scipy.stats import t
import math

# Parameters
m = 10
mean_interarrival = 1
mean_service = 8
n_customers = 10000
n_replications = 10

# Pareto shape parameters
pareto_shapes = [1.05, 2.05]

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system_pareto_service(
    n_customers, m, mean_interarrival, mean_service, pareto_shape, rng
):
    """
    Simulates a blocking system with Poisson arrivals and Pareto service times.
    The Pareto scale parameter is chosen such that the mean service time is 8.
    """

    # Poisson arrivals: exponential inter-arrival times
    interarrival_times = rng.exponential(mean_interarrival, size=n_customers)
    arrival_times = np.cumsum(interarrival_times)

    # Choose Pareto scale so the mean is equal to mean_service
    xm = mean_service * (pareto_shape - 1) / pareto_shape

    # Pareto service times
    service_times = xm * (rng.pareto(pareto_shape, size=n_customers) + 1)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If the server is free, the customer is served
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    return blocked / n_customers


for pareto_shape in pareto_shapes:

    blocking_fractions = []

    for i in range(n_replications):
        fraction = simulate_blocking_system_pareto_service(
            n_customers,
            m,
            mean_interarrival,
            mean_service,
            pareto_shape,
            rng
        )
        blocking_fractions.append(fraction)

    blocking_fractions = np.array(blocking_fractions)

    # Estimate mean blocking fraction
    mean_blocking = np.mean(blocking_fractions)

    # Confidence interval
    sample_sd = np.std(blocking_fractions, ddof=1)
    alpha = 0.05
    t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

    lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
    upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

    xm = mean_service * (pareto_shape - 1) / pareto_shape

    print("\nPareto shape k =", pareto_shape)
    print("Pareto scale xm =", xm)
    print("Blocking fractions from the 10 replications:")
    print(blocking_fractions)

    print("Estimated blocking fraction:")
    print(mean_blocking)

    print("95% confidence interval:")
    print((lower, upper))


Pareto shape k = 1.05
Pareto scale xm = 0.38095238095238126
Blocking fractions from the 10 replications:
[0.0006 0.0001 0.0042 0.0013 0.0007 0.0019 0.0003 0.0012 0.0024 0.0006]
Estimated blocking fraction:
0.00133
95% confidence interval:
(np.float64(0.0004454415881556713), np.float64(0.0022145584118443288))

Pareto shape k = 2.05
Pareto scale xm = 4.097560975609756
Blocking fractions from the 10 replications:
[0.1397 0.1246 0.1093 0.1151 0.1292 0.1153 0.1115 0.1152 0.1138 0.1169]
Estimated blocking fraction:
0.11905999999999999
95% confidence interval:
(np.float64(0.11236214691083965), np.float64(0.12575785308916032))


### c) Choose one or two other distributions.

In [7]:
import numpy as np
from scipy.stats import t
import math

# Parameters
m = 10
mean_interarrival = 1
mean_service = 8
n_customers = 10000
n_replications = 10

# Set seed for reproducibility
rng = np.random.default_rng(12345)


def simulate_blocking_system_general_service(
    n_customers, m, mean_interarrival, service_distribution, rng
):
    """
    Simulates a blocking system with Poisson arrivals and a chosen service time distribution.
    The system has m service units and no waiting room.
    """

    # Poisson arrivals: exponential inter-arrival times
    interarrival_times = rng.exponential(mean_interarrival, size=n_customers)
    arrival_times = np.cumsum(interarrival_times)

    # Generate service times from the chosen service distribution
    service_times = service_distribution(rng, n_customers)

    # End time for each service unit
    server_end_times = np.zeros(m)

    blocked = 0

    for arrival_time, service_time in zip(arrival_times, service_times):

        # Find the server that becomes free first
        next_free_server = np.argmin(server_end_times)

        # If the server is free, the customer is served
        if server_end_times[next_free_server] <= arrival_time:
            server_end_times[next_free_server] = arrival_time + service_time

        # Otherwise, all servers are busy and the customer is blocked
        else:
            blocked += 1

    return blocked / n_customers


def run_replications(service_distribution):
    """
    Runs 10 independent replications and returns the mean blocking fraction
    and a 95% confidence interval.
    """

    blocking_fractions = []

    for i in range(n_replications):
        fraction = simulate_blocking_system_general_service(
            n_customers,
            m,
            mean_interarrival,
            service_distribution,
            rng
        )
        blocking_fractions.append(fraction)

    blocking_fractions = np.array(blocking_fractions)

    mean_blocking = np.mean(blocking_fractions)

    sample_sd = np.std(blocking_fractions, ddof=1)
    alpha = 0.05
    t_value = t.ppf(1 - alpha / 2, df=n_replications - 1)

    lower = mean_blocking - t_value * sample_sd / np.sqrt(n_replications)
    upper = mean_blocking + t_value * sample_sd / np.sqrt(n_replications)

    return blocking_fractions, mean_blocking, lower, upper


# Service distribution 1: Uniform(4, 12), mean = 8
def uniform_service(rng, n):
    return rng.uniform(4, 12, size=n)


# Service distribution 2: Gamma(shape = 2, scale = 4), mean = 8
def gamma_service(rng, n):
    return rng.gamma(shape=2, scale=4, size=n)


# Run simulations
uniform_results = run_replications(uniform_service)
gamma_results = run_replications(gamma_service)


# Print results
print("Uniform service times U(4, 12)")
print("Blocking fractions:")
print(uniform_results[0])
print("Estimated blocking fraction:")
print(uniform_results[1])
print("95% confidence interval:")
print((uniform_results[2], uniform_results[3]))

print("\nGamma service times Gamma(2, 4)")
print("Blocking fractions:")
print(gamma_results[0])
print("Estimated blocking fraction:")
print(gamma_results[1])
print("95% confidence interval:")
print((gamma_results[2], gamma_results[3]))


# Erlang B formula for comparison
A = mean_service / mean_interarrival

def erlang_b(A, m):
    numerator = A**m / math.factorial(m)
    denominator = sum(A**i / math.factorial(i) for i in range(m + 1))
    return numerator / denominator

exact_blocking = erlang_b(A, m)

print("\nExact Erlang B blocking probability:")
print(exact_blocking)

Uniform service times U(4, 12)
Blocking fractions:
[0.1251 0.1237 0.124  0.1299 0.1238 0.1236 0.1234 0.1266 0.1226 0.1223]
Estimated blocking fraction:
0.12450000000000001
95% confidence interval:
(np.float64(0.12288660980390588), np.float64(0.12611339019609413))

Gamma service times Gamma(2, 4)
Blocking fractions:
[0.1228 0.125  0.1206 0.1234 0.1195 0.1288 0.1199 0.1181 0.1191 0.1236]
Estimated blocking fraction:
0.12208
95% confidence interval:
(np.float64(0.11973665557304677), np.float64(0.12442334442695321))

Exact Erlang B blocking probability:
0.1216610642529515


## Part 5 

Compare confidence intervals for Parts 1, 2, and 3 then interpret and explain differences if any.


In [8]:
import pandas as pd

# Results from the previous parts
results = {
    "Part": [
        "Part 1",
        "Part 2a",
        "Part 2b",
        "Part 3a",
        "Part 3b, k = 1.05",
        "Part 3b, k = 2.05",
        "Part 3c, Uniform",
        "Part 3c, Gamma"
    ],
    "Description": [
        "Poisson arrivals, exponential service",
        "Erlang inter-arrival times, exponential service",
        "Hyperexponential inter-arrival times, exponential service",
        "Poisson arrivals, constant service",
        "Poisson arrivals, Pareto service",
        "Poisson arrivals, Pareto service",
        "Poisson arrivals, uniform service",
        "Poisson arrivals, gamma service"
    ],
    "Mean blocking fraction": [
        0.12355,
        0.09107,
        0.13650,
        0.12166,
        0.00133,
        0.12447,
        0.12450,
        0.12132
    ],
    "Lower 95% CI": [
        0.12015,
        0.08852,
        0.13240,
        0.11909,
        0.00045,
        0.11917,
        0.12289,
        0.11721
    ],
    "Upper 95% CI": [
        0.12695,
        0.09362,
        0.14060,
        0.12423,
        0.00221,
        0.12977,
        0.12611,
        0.12543
    ]
}

df = pd.DataFrame(results)

# Add width of confidence interval
df["CI width"] = df["Upper 95% CI"] - df["Lower 95% CI"]

print(df)

                Part                                        Description  \
0             Part 1              Poisson arrivals, exponential service   
1            Part 2a    Erlang inter-arrival times, exponential service   
2            Part 2b  Hyperexponential inter-arrival times, exponent...   
3            Part 3a                 Poisson arrivals, constant service   
4  Part 3b, k = 1.05                   Poisson arrivals, Pareto service   
5  Part 3b, k = 2.05                   Poisson arrivals, Pareto service   
6   Part 3c, Uniform                  Poisson arrivals, uniform service   
7     Part 3c, Gamma                    Poisson arrivals, gamma service   

   Mean blocking fraction  Lower 95% CI  Upper 95% CI  CI width  
0                 0.12355       0.12015       0.12695   0.00680  
1                 0.09107       0.08852       0.09362   0.00510  
2                 0.13650       0.13240       0.14060   0.00820  
3                 0.12166       0.11909       0.12423   0.00

## Exact solution

Consider a system with a Poisson arrival process with intensity $\lambda$ and a general service time distribution with mean service time $s$. Define then the offered traffic $A = \lambda s$.

The Erlang's B-formula applies:

$B = P(m) = \frac{\frac{A^m}{m!}}{\sum_{i=0}^{m} \frac{A^i}{i!}}$.